# Mini-Transformer causal · misma pregunta, otra máquina

**Meta.** Next-token otra vez. El objetivo $\mathcal{L}$ no cambió. Cambió *quién mira el pasado*.

La GRU aplastaba todo en un post-it $h_t$. Aquí cada letra voltea a ver el pasado con **pesos**. Eso es atención.

## Dummy, pieza por pieza

| Pieza | En cristiano |
|---|---|
| Embedding | cada letra se vuelve un vector |
| Q, K, V | qué busco / qué ofrezco / qué entrego |
| Atención | de lo que ya leí, ¿qué me importa *ahora*? |
| Máscara causal | no veas el futuro (si no, haces trampa) |
| Multi-head | varias miradas a la vez (sujeto, coma, rima…) |
| Posición | sin esto, “perro muerde hombre” = “hombre muerde perro” |
| FFN | cada posición cocina lo que ya vio; no mezcla tiempo |
| Residual $x+f(x)$ | si $f$ se equivoca, el camino original sigue ahí |

Un bloque, en el pizarrón:

```
x = x + CausalMHA(LayerNorm(x))
x = x + FFN(LayerNorm(x))
```

Apilas 2 (CPU) o 4 (GPU) y ya tienes un mini-GPT.

## Matemática · atención (Vaswani et al., 2017)

Con consultas, claves y valores $Q,K,V\in\mathbb{R}^{T\times d_k}$,

$$
\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V.
$$

$\sqrt{d_k}$ es fontanería: sin eso, con vectores largos los productos se van a mil y el softmax se vuelve un one-hot inútil.

Q, K, V no son tres almas distintas. Son tres proyecciones del mismo $x$. Estoy en “negó” (Q busca un sujeto). “presidente” ofrece una pista (K). El producto $QK^\top$ mide el encaje. Softmax lo vuelve pesos que suman 1. V es lo que entrego si me eligen.

**Multi-head:** $h$ copias en paralelo, cada una con sus $W_i^Q,W_i^K,W_i^V$, y al final se concatenan. No es otro algoritmo: es la misma atención, repetida.


## Máscara causal · la trampa #1 de este lab

**Dummy.** Entrenar un generador sin máscara es hacer trampa en el examen. El token de la posición 5 podría “ver” la letra 6, que es precisamente la respuesta. La val_loss baja como loca y cuando generas de verdad —donde el futuro *no existe*— el modelo se cae.

**Math.** Se suma $M\in\{0,-\infty\}^{T\times T}$ a los logits de atención:

$$
M_{ij}=\begin{cases}0&j\le i\\-\infty&j>i\end{cases}
$$

Tras el softmax, $A_{ij}=0$ si $j>i$: el token $i$ no ve el futuro. En el código es `torch.tril` + `masked_fill(..., -inf)`.

**Pregunta de salida:** ¿qué pasa si comentas la máscara? (Hazlo una vez, anota val_loss vs generación.)

## Posición · la atención desnuda no sabe de orden

Si barajas las palabras, los productos internos salen igual. Por eso sumamos un embedding posicional $p_t$ distinto en cada sitio: se atiende sobre $x_t+p_t$. (Qwen usa RoPE, una rotación según la distancia $i-j$. Misma necesidad, otro truco.)

$$
\mathrm{Attention}(\pi Q,\pi K,\pi V)=\pi\,\mathrm{Attention}(Q,K,V)
$$

si $\pi$ permuta filas: la atención es permutación-invariante. El orden se inyecta a mano.

Decoder causal (tipo GPT) $\neq$ encoder bidireccional (tipo BERT). BERT ve ambos lados; no sirve tal cual para generar.


In [ ]:
# Colab: descomenta las dos líneas de abajo (una vez).
# %pip install -q torch
# !wget -q https://raw.githubusercontent.com/NoSoyRo/diplomado-ia-llms/main/datos/quijote.txt

from __future__ import annotations

import math
from dataclasses import dataclass
from typing import List, Optional, Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
from pathlib import Path

# Local: ../datos/quijote.txt (o ../../ desde extra/). Colab: el wget lo deja junto al notebook.
TEXT_PATH = next(
    (str(p) for p in (Path("quijote.txt"), Path("../datos/quijote.txt"), Path("../../datos/quijote.txt")) if p.exists()),
    "quijote.txt",
)
print(f"device={DEVICE}  archivo={TEXT_PATH}")


In [ ]:
def load_text(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read().replace("\r\n", "\n").replace("\r", "\n")


def build_vocab(text: str):
    chars = sorted(set(text))
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for ch, i in stoi.items()}
    return stoi, itos


def encode(text: str, stoi: dict) -> torch.Tensor:
    return torch.tensor([stoi[ch] for ch in text], dtype=torch.long)


def decode(ids: List[int], itos: dict) -> str:
    return "".join(itos[i] for i in ids)


In [ ]:
class CharWindowDataset(Dataset):
    """Ventanas x → y corrido +1. Si max_windows no es None, muestrea inicios al azar.

    Dummy: el Quijote tiene ~2M letras. Recorrer TODAS las ventanas en CPU no cabe
    en una sesión. Muestrear M inicios es el mismo examen, con menos filas.
    """

    def __init__(self, data_ids: torch.Tensor, seq_len: int, max_windows: Optional[int] = 8000, seed: int = 0):
        self.data = data_ids
        self.seq_len = seq_len
        n = max(0, len(self.data) - self.seq_len - 1)
        if max_windows is not None and n > max_windows:
            g = torch.Generator().manual_seed(seed)
            self.index = torch.randint(0, n, (max_windows,), generator=g)
        else:
            self.index = torch.arange(n)

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i: int):
        idx = int(self.index[i])
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + 1 : idx + self.seq_len + 1]
        return x, y


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd: int, n_head: int, dropout: float = 0.1):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = torch.tril(torch.ones(T, T, device=x.device)).view(1, 1, T, T)
        att = att.masked_fill(mask == 0, float("-inf"))
        att = torch.softmax(att, dim=-1)
        att = self.dropout(att)

        out = att @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.dropout(self.proj(out))


class MLP(nn.Module):
    def __init__(self, n_embd: int, dropout: float = 0.1):
        super().__init__()
        self.fc1 = nn.Linear(n_embd, 4 * n_embd)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.fc2(self.act(self.fc1(x))))


class Block(nn.Module):
    def __init__(self, n_embd: int, n_head: int, dropout: float = 0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class MiniTransformerLM(nn.Module):
    def __init__(self, vocab_size: int, seq_len: int, n_embd: int = 128, n_head: int = 4, n_layer: int = 2, dropout: float = 0.1):
        super().__init__()
        self.seq_len = seq_len
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(seq_len, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)

    def forward(self, idx):
        B, T = idx.size()
        assert T <= self.seq_len, "T excede seq_len del modelo"
        tok = self.tok_emb(idx)
        pos = self.pos_emb(torch.arange(T, device=idx.device))
        x = self.drop(tok + pos.unsqueeze(0))
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.ln_f(x))


## Entrenar y generar

Misma $\mathcal{L}$ del lab 0. AdamW en vez de Adam (decaimiento de pesos: un empujón extra para que no se hinchen). Al generar, recortamos a las últimas `seq_len` letras: el modelo no tiene vaso infinito, tiene una **ventana**.

CPU: 2 capas × 128. GPU: 4 × 256. Cuenta parámetros (`params=…M`) y compáralos con la GRU: más carne, más mirada, más VRAM.


In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    criterion = nn.CrossEntropyLoss()
    total = 0.0
    n = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        B, T, V = logits.shape
        loss = criterion(logits.view(B * T, V), y.view(B * T))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
        n += 1
    return total / max(1, n)


@torch.no_grad()
def eval_loss(model, loader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total = 0.0
    n = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        B, T, V = logits.shape
        loss = criterion(logits.view(B * T, V), y.view(B * T))
        total += loss.item()
        n += 1
    return total / max(1, n)


@torch.no_grad()
def generate(model, prompt: str, stoi: dict, itos: dict, seq_len: int, max_new_tokens: int, temperature: float, device: str):
    model.eval()
    device = torch.device(device)
    ids = [stoi[ch] for ch in prompt if ch in stoi] or [0]
    out = ids[:]
    for _ in range(max_new_tokens):
        ctx = out[-seq_len:]
        x = torch.tensor(ctx, dtype=torch.long, device=device).unsqueeze(0)
        logits = model(x)[0, -1, :] / max(1e-8, temperature)
        next_id = torch.multinomial(torch.softmax(logits, dim=-1), 1).item()
        out.append(next_id)
    return decode(out, itos)


In [ ]:
@dataclass
class TrainConfig:
    seq_len: int = 64
    batch_size: int = 32
    n_embd: int = 128
    n_head: int = 4
    n_layer: int = 2
    dropout: float = 0.1
    lr: float = 3e-4
    epochs: int = 3
    max_windows: int = 6000
    device: str = DEVICE
    temperature_list: Tuple[float, ...] = (0.7, 1.0, 1.2)
    gen_tokens: int = 300
    prompt: str = "En un lugar de la Mancha "


def main(text_path: str, cfg: TrainConfig):
    text = load_text(text_path)
    stoi, itos = build_vocab(text)
    data_ids = encode(text, stoi)
    n = len(data_ids)
    split = int(0.9 * n)
    train_ds = CharWindowDataset(data_ids[:split], cfg.seq_len, cfg.max_windows, seed=0)
    val_ds = CharWindowDataset(data_ids[split:], cfg.seq_len, min(1500, cfg.max_windows), seed=1)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, drop_last=True)

    device = torch.device(cfg.device)
    model = MiniTransformerLM(len(stoi), cfg.seq_len, cfg.n_embd, cfg.n_head, cfg.n_layer, cfg.dropout).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Vocab={len(stoi)} | params={n_params/1e6:.2f}M | L={cfg.seq_len} | layers={cfg.n_layer} heads={cfg.n_head}")

    for epoch in range(1, cfg.epochs + 1):
        tr = train_one_epoch(model, train_loader, optimizer, device)
        vl = eval_loss(model, val_loader, device)
        ppl = math.exp(vl) if vl < 20 else float("inf")
        print(f"Epoch {epoch:02d} | train_loss={tr:.4f} | val_loss={vl:.4f} | val_ppl={ppl:.2f}")

    for temp in cfg.temperature_list:
        sample = generate(model, cfg.prompt, stoi, itos, cfg.seq_len, cfg.gen_tokens, temp, cfg.device)
        print("\n" + "=" * 70)
        print(f"Temperature = {temp}")
        print(sample[:2000])


cfg = TrainConfig()
print("CPU: 2 capas × 128. GPU: n_layer=4, n_embd=256, seq_len=128, epochs=5.")
main(TEXT_PATH, cfg)
